# Step 1 — Project setup, recording manifest, and Ilastik reference images

This notebook creates or loads the analysis directory, discovers raw recording folders, builds an auditable manifest, and exports one reference BMP per configured wavelength and recording. Run the cells **from top to bottom**.

**Geometry and identity:** Step1 reference BMPs are rotated at full spatial resolution (`rotated_fullres`). The later `raw.scale` setting is **not** applied here. The recording `stem` uses the configured ordered `metadata.stem_keys`, including the full `protocol` value. Wavelengths are selected by the positional `raw.wave_pattern`, not inferred from image pixels.

**Before running:** Set `CONFIG_PATH` in the code cell below to your actual `config.yaml`. The notebook contains an example machine-specific path that must be adapted. The code assumes a notebook working directory from which `src/` can be located (current directory, its `src/`, or the parent directory's `src/`).

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import imageio.v3 as iio
import matplotlib.pyplot as plt

# Expected layout:
# project/notebook/Step1_ProjectCreation.ipynb
# project/src/common.py + step1_project_creation.py

NOTEBOOK_DIR = Path.cwd().resolve()
SRC_DIR = NOTEBOOK_DIR.parent / "src"

if not (SRC_DIR / "common.py").is_file() or not (SRC_DIR / "step1_project_creation.py").is_file():
    raise FileNotFoundError(
        "Could not find common.py and step1_project_creation.py in the project's src/ directory. "
        f"Expected location: {SRC_DIR}"
    )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from common import load_config, create_or_load_project, build_standard_paths, get_stem_keys
from step1_project_creation import (
    discover_recording_folders, build_recording_manifest,
    save_manifest, save_reference_images_for_manifest,
)
print('Source folder:', SRC_DIR)


## 1. Load the configuration and initialize the project

Set `CONFIG_PATH` to your YAML configuration file. The file can be stored anywhere, provided that its path is specified correctly.

**For a new analysis project**, configure the following settings in `config.yaml`:

```yaml
project:
  directory_to_save_main_analysis: null
  analysis_parent_dir: "F:/Analysis"
  analysis_folder_name: "My_experiment"
  create_new_analysis_folder: true
```

- `analysis_parent_dir` — the parent directory where the new analysis folder will be created.
- `analysis_folder_name` — the name of the new analysis folder.
- `create_new_analysis_folder: true` — creates a dedicated analysis folder. If the target folder already exists and is not empty, an error is raised.
- `directory_to_save_main_analysis: null` — indicates that the analysis directory has not yet been assigned.

The notebook loads the configuration using `load_config()` and initializes the analysis directory using `create_or_load_project()`.

On the first run, `create_or_load_project()` automatically writes the full analysis directory path to `project.directory_to_save_main_analysis` in the original YAML file. If `config_snapshot_name` is configured, an initial configuration snapshot is also saved in the analysis directory.

`build_standard_paths()` provides the output locations used throughout the pipeline.

**Important:** If `directory_to_save_main_analysis` already contains a path, that directory is reused instead of creating a new analysis folder. Leave this field `null` only when initializing a new project.

In [ ]:
# Configure the project YAML path explicitly; it need not match the notebook directory.
CONFIG_PATH = Path(r"config.yaml")
config = load_config(CONFIG_PATH)
stem_keys = get_stem_keys(config)
project_paths = create_or_load_project(config, CONFIG_PATH)
paths = build_standard_paths(config, project_paths)
print('config:', CONFIG_PATH)
print('stem_keys:', stem_keys)
print('analysis_root:', project_paths['analysis_root'])
print('manifest_path:', paths['manifest_path'])
print('reference_images_root:', paths['reference_images_root'])


## 2. Check the project configuration before discovery

Review these YAML settings before running the remaining cells:

- `discovery.raw_root`, `discovery.mode`, `discovery.suffixes`, and `discovery.min_files` (for recursive discovery).
- `metadata.regex`, `metadata.stem_keys`, and `discovery.metadata_filters` (which determine recording identity and initial inclusion).
- `raw.file_regex`, `raw.sort_key`, `raw.wave_pattern`, and `raw.ref_image_idx_per_wavelength` (which determine the selected reference frame per wavelength).
- `raw.rotation` and `raw.scale`: reference BMPs use **rotation only**; `raw.scale` is recorded for downstream raw-Zarr processing.
- `project.directory_to_save_main_analysis` and `project.create_new_analysis_folder` (which control analysis-root selection).

**Configuration-specific note:** The supplied example YAML permits both `Awake` and `Iso` in `discovery.metadata_filters.state`; it is not an Awake-only selection. The example sets `raw.rotation: -90.0` degrees and `raw.scale: 0.3`.

Use a new analysis directory for a new project. Do not reuse old masks or registration outputs unless that reuse is intentionally part of the analysis.

## 3. Discover recording folders

The configured discovery strategy finds recording directories beneath `discovery.raw_root`. With `recursive_folders`, a directory is selected when it contains at least `discovery.min_files` eligible files; once selected, nested folders are not searched further. Review the resulting count and example paths before building the manifest.

In [ ]:
recording_dirs = discover_recording_folders(config)
print("n recording folders:", len(recording_dirs))
recording_dirs[:6]

## 4. Build and inspect the recording manifest

`build_recording_manifest` parses path metadata, applies `discovery.metadata_filters`, checks for duplicate `stem` values, and writes `manifest_preflight.csv`. If any folder causes an error, it writes `manifest_errors.csv` and raises an exception. The accepted manifest is saved later, after reference-image paths have been added.

The following cells show recording identity fields, inclusion/exclusion statuses, and a summary grouped by configured stem keys.

In [ ]:
manifest_df = build_recording_manifest(config, paths)
preflight_path = paths['manifest_path'].with_name('manifest_preflight.csv')
print('manifest rows:', len(manifest_df))
print('preflight:', preflight_path)
cols_to_show = ['stem'] + [k for k in stem_keys if k in manifest_df.columns] + ['protocol_base', 'img_dir']
cols_to_show = list(dict.fromkeys(c for c in cols_to_show if c in manifest_df.columns))
display(manifest_df[cols_to_show].head())


In [ ]:
if not manifest_df.empty and 'state' in manifest_df:
    print('states:', manifest_df['state'].dropna().unique().tolist())
if preflight_path.exists():
    audit_df = pd.read_csv(preflight_path)
    display(audit_df['status'].value_counts(dropna=False).rename_axis('status').reset_index(name='n_recordings'))
    display(audit_df[audit_df['status'].ne('included')].head(30))


In [ ]:
# Summary by all stem_keys that exist in manifest
#summary_cols = [k for k in stem_keys if k in manifest_df.columns]
summary_cols = ['exp_day', 'state']
if summary_cols and not manifest_df.empty:
    display(manifest_df.groupby(summary_cols, dropna=False).size().reset_index(name="n_recordings"))
else:
    print("No summary columns available. Check metadata.regex and metadata.stem_keys.")

In [ ]:
summary_cols

## 5. Export full-resolution reference BMPs for Ilastik

**Proceed only after inspecting the manifest preflight results.** The exporter sorts the raw BMP files, selects a positional wavelength subsequence using `raw.wave_pattern`, and selects one reference image per wavelength using `raw.ref_image_idx_per_wavelength`. It applies `raw.rotation` but not `raw.scale` and records the expected Ilastik probability H5 path in the manifest.

The cell then saves `manifest_recordings.csv`. Existing reference BMPs may be replaced according to `step1.reference_selection.overwrite`; the manifest itself is saved with `overwrite=True` by this notebook.

In [ ]:
# Save reference BMPs only after reviewing the preflight results above.
if manifest_df.empty:
    raise RuntimeError('Manifest пуст. Reference BMP не создаются.')
manifest_df = save_reference_images_for_manifest(
    manifest_df=manifest_df, config=config, paths=paths, show_progress=True,
)
save_manifest(manifest_df, paths['manifest_path'], overwrite=True)
print('Saved manifest:', paths['manifest_path'])
ref_cols = [c for c in manifest_df.columns if c.startswith('ref_') or c.startswith('prob_')]
display(manifest_df[['stem'] + ref_cols].head())


## 6. Check expected Ilastik probability H5 files for Step2

The manifest records paths such as `<stem>_Probabilities.h5`, which are expected **after** Ilastik segmentation and export. Their absence on the first Step1 run is normal and does not mean that reference-BMP generation failed.

The following cell checks current file existence for display; it does not generate probability H5 files.

In [ ]:
prob_cols = [c for c in manifest_df.columns if c.startswith("prob_") and c.endswith("_path")]
for col in prob_cols:
    exists_col = col.replace("_path", "_exists_now")
    manifest_df[exists_col] = manifest_df[col].map(lambda p: Path(str(p)).exists())

display(manifest_df[["stem"] + prob_cols + [c.replace("_path", "_exists_now") for c in prob_cols]].head())

## 7. Preview selected reference images

The following preview displays wavelengths `505` and `656` for the first manifest row. These two wavelengths are explicitly selected in the existing preview cell and correspond to the supplied example configuration; other wavelength configurations require a different preview selection. Check orientation, framing, and the presence of both reference images.

In [ ]:
if not manifest_df.empty:
    row = manifest_df.iloc[0]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, wl in zip(axes, [ "505", "656"]):
        p = Path(row[f"ref_{wl}_path"])
        if p.exists():
            ax.imshow(iio.imread(p), cmap="gray")
            ax.set_title(wl)
        else:
            ax.set_title(f"{wl}: missing")
        ax.axis("off")
    plt.tight_layout()
else:
    print("Manifest is empty.")

## Step1 outputs and hand-off to Step2

Confirm the recording cohort and source directories in `manifest_preflight.csv` and `manifest_recordings.csv`. Inspect the size and orientation of the exported reference BMPs under the Step1 Ilastik reference-image directory.

Next, run Ilastik segmentation and export the probability H5 files at the expected manifest paths. Step2 uses these exports to construct masks and perform registration. Do not copy earlier projects' masks or registration outputs into this new analysis directory.

In [ ]:
print('New project:', project_paths['analysis_root'])
print('Recordings in manifest:', len(manifest_df))
print('Manifest:', paths['manifest_path'])
print('Preflight:', preflight_path)
print('Reference BMP folder:', paths['reference_images_root'])
print('Next: Ilastik → Step2')
